In [ ]:
import torch
from torch.autograd import Variable
from mpc import mpc
from mpc.mpc import QuadCost, LinDx, GradMethods
from mpc.env_dx import pendulum

params = torch.tensor((10., 1., 1.)) # Gravity, mass, length.
dx = pendulum.PendulumDx(params, simple=True)

# --- 缺失参数 ---
mpc_T   = 20                     # MPC 规划步数（时间域长度）
n_batch = 1                      # 批次大小（并行仿真数）

# 初始状态：摆处于竖直向下位置 (cos θ=-1, sin θ=0, θ̇=0)
# 目标状态 goal_state=(1,0,0) 表示竖直向上 (cos θ=1)
x = torch.tensor([[-1., 0., 0.]])  # shape: (n_batch, n_state)

# 初始控制序列：全零，shape: (T, n_batch, n_ctrl)
u_init = torch.zeros(mpc_T, n_batch, dx.n_ctrl)

In [ ]:
goal_weights = torch.Tensor((1., 1., 0.1))
goal_state = torch.Tensor((1., 0. ,0.))
ctrl_penalty = 0.001
q = torch.cat((
    goal_weights,
    ctrl_penalty*torch.ones(dx.n_ctrl)
))

px = -torch.sqrt(goal_weights)*goal_state
p = torch.cat((px, torch.zeros(dx.n_ctrl)))
Q = torch.diag(q).unsqueeze(0).unsqueeze(0).repeat(
    mpc_T, n_batch, 1, 1
)
p = p.unsqueeze(0).repeat(mpc_T, n_batch, 1)

nominal_states, nominal_actions, nominal_objs = mpc.MPC(
    dx.n_state, dx.n_ctrl, mpc_T,
    u_init=u_init,
    u_lower=dx.lower, u_upper=dx.upper,
    lqr_iter=50,
    verbose=0,
    exit_unconverged=False,
    detach_unconverged=False,
    linesearch_decay=dx.linesearch_decay,
    max_linesearch_iter=dx.max_linesearch_iter,
    grad_method=GradMethods.AUTO_DIFF,
    eps=1e-2,
)(x, QuadCost(Q, p), dx)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ---- 闭环滚动 MPC 仿真 ----
n_sim   = 100          # 仿真总步数
state   = x.clone()   # 当前状态
ctrl    = u_init.clone()

log_states  = []       # 记录实际轨迹
log_actions = []

for t in range(n_sim):
    nom_states, nom_actions, nom_obj = mpc.MPC(
        dx.n_state, dx.n_ctrl, mpc_T,
        u_init=ctrl,
        u_lower=dx.lower, u_upper=dx.upper,
        lqr_iter=50,
        verbose=0,
        exit_unconverged=False,
        detach_unconverged=False,
        linesearch_decay=dx.linesearch_decay,
        max_linesearch_iter=dx.max_linesearch_iter,
        grad_method=GradMethods.AUTO_DIFF,
        eps=1e-2,
    )(state, QuadCost(Q, p), dx)

    # 取第一步动作，执行到环境
    action = nom_actions[0]                         # (n_batch, n_ctrl)
    state  = dx(state, action)                      # 推进一步

    log_states.append(state.detach().squeeze(0).numpy())
    log_actions.append(action.detach().squeeze(0).numpy())

    # 温启动：将剩余计划序列前移，最后一步复制
    ctrl = torch.cat([nom_actions[1:], nom_actions[-1:]], dim=0).detach()

log_states  = np.array(log_states)   # (n_sim, 3)
log_actions = np.array(log_actions)  # (n_sim, 1)

# ---- 绘图 ----
t_axis = np.arange(n_sim) * dx.dt

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

axes[0].plot(t_axis, log_states[:, 0], label='cos θ')
axes[0].plot(t_axis, log_states[:, 1], label='sin θ')
axes[0].axhline(1.0, color='gray', linestyle='--', linewidth=0.8, label='goal (cos θ=1)')
axes[0].set_ylabel('cos θ / sin θ')
axes[0].legend()
axes[0].set_title('Pendulum Swing-Up via MPC (closed-loop)')

axes[1].plot(t_axis, log_states[:, 2], color='C2', label='θ̇  (rad/s)')
axes[1].set_ylabel('θ̇  (rad/s)')
axes[1].legend()

axes[2].plot(t_axis, log_actions[:, 0], color='C3', label='torque')
axes[2].axhline( dx.upper, color='gray', linestyle=':', linewidth=0.8)
axes[2].axhline(-dx.upper, color='gray', linestyle=':', linewidth=0.8)
axes[2].set_ylabel('Control (Nm)')
axes[2].set_xlabel('Time (s)')
axes[2].legend()

plt.tight_layout()
plt.show()

# 打印最终状态与目标
final = log_states[-1]
print(f"Final state : cos θ={final[0]:.3f}, sin θ={final[1]:.3f}, θ̇={final[2]:.3f}")
print(f"Goal  state : cos θ=1.000, sin θ=0.000, θ̇=0.000")